# v3 Pair Generation: DPO preference pairs from v2.1-lite

Samples N=8 completions per in-distribution transcript from v2.1-lite,
scores each with the verifier, and builds (chosen, rejected) pairs for DPO training.

**Problem targeted:** Problem 2 (behavioural misses on in-distribution data).
Pairs are drawn from templates v2.1 was trained on (soap, referral_a, mse) only.

**Runs on:** Kaggle GPU (T4 or P100).
**Output:** `data/qwen3.5_latest/dpo_pairs.jsonl`

## Cell 1: env check

In [ ]:
import sys
print('Python:', sys.version)
!nvidia-smi

## Cell 2: clone repo

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient

PAT  = UserSecretsClient().get_secret('GITHUB_PAT')
REPO = 'nizamphoenix/clinical_transcript_summariser'
DEST = '/kaggle/working/repo'

if os.path.exists(DEST):
    subprocess.run(['rm', '-rf', DEST], check=True)

url = f'https://{PAT}@github.com/{REPO}.git'
subprocess.run(['git', 'clone', '--depth', '1', url, DEST], check=True)
print('cloned ->', DEST)
%cd $DEST

## Cell 3: install dependencies

In [ ]:
!pip install -q unsloth
!pip uninstall unsloth -y
!pip install -q --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
!pip install -q -e .

import sys
sys.path.insert(0, '/kaggle/working/repo')
print('install done')

## Cell 4: imports and data loading

In [ ]:
import json, random, time
from pathlib import Path

from src.verifier import score_prediction, reward
from src.prompts import build_inference_messages
from src.data_generation.templates import REGISTRY

DATA_ROOT = Path('data/qwen3.5_latest')

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

# In-distribution train transcripts only (Problem 2 targets trained templates)
rows = (
    [dict(r, template='soap')       for r in load_jsonl(DATA_ROOT / 'train.soap.jsonl')] +
    [dict(r, template='referral_a') for r in load_jsonl(DATA_ROOT / 'train.referral_a.jsonl')] +
    [dict(r, template='mse')        for r in load_jsonl(DATA_ROOT / 'train.mse.jsonl')]
)
random.Random(42).shuffle(rows)
print(f'total transcripts: {len(rows)}')

# Verify every row has template field and its label key
for r in rows:
    lk = REGISTRY[r['template']]['label_key']
    assert lk in r, f"missing {lk} in row with template={r['template']}"
print('label key check passed')

## Cell 5: load v2.1 model for sampling

In [ ]:
from unsloth import FastLanguageModel

MODEL_PATH = '/kaggle/input/v2-1-lite-multi-template-sft-unsloth/merged_fp16_v2_1_lite'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
print('model loaded')

## Cell 6: sampling function

In [ ]:
import torch

N_SAMPLES    = 8
TEMPERATURE  = 0.9
TOP_P        = 0.95
MAX_NEW_TOKENS = 1024

def sample_n(template: str, transcript: str, n: int = N_SAMPLES) -> list[str]:
    """Generate n completions for a single transcript + template."""
    messages = build_inference_messages(template, transcript)
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        num_return_sequences=n,
        pad_token_id=tokenizer.eos_token_id,
    )
    # Decode only the newly-generated tokens for each sequence
    input_len = inputs['input_ids'].shape[1]
    results = []
    for seq in outputs:
        new_tokens = seq[input_len:]
        results.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())
    return results

# Smoke test on first row
smoke = rows[0]
smoke_samples = sample_n(smoke['template'], smoke['transcript'], n=2)
print(f'smoke test ok: generated {len(smoke_samples)} samples')
print('sample[0] (first 200 chars):', smoke_samples[0][:200])

## Cell 7: pair construction loop

In [ ]:
QUALITY_THRESHOLD = 0.5
MIN_GAP           = 0.2
LARGE_GAP         = 0.6   # second rejected pair added when gap exceeds this
OUT_PATH          = DATA_ROOT / 'dpo_pairs.jsonl'

BUCKETS = ('miss', 'over_populated', 'hallucination', 'schema_invalid')


def classify_rejected_mode(sc: dict) -> str:
    """Label the dominant failure mode for a rejected sample.

    Priority order (most specific first):
      schema_invalid  -> hard schema failure
      over_populated  -> fills gold-null fields (Problem 2b)
      hallucination   -> ungrounded spans (Problem 2b)
      miss            -> content miss / wrong_null (Problem 2a)
    """
    if not sc['schema_valid']:
        return 'schema_invalid'
    if sc.get('over_populated', 0) > 0:
        return 'over_populated'
    total = sc.get('total_spans', 0)
    grounded = sc.get('grounded_spans', 0)
    if total > 0 and (grounded / total) < 0.8:
        return 'hallucination'
    return 'miss'


def pick_rejected_idx(scores, rewards, exclude_idx, bucket_counts):
    """Pick the rejected sample index that balances bucket representation.

    Prefers the sample whose failure bucket is most under-represented globally.
    Falls back to worst-reward if no valid rejected candidates remain.
    """
    candidates = [
        j for j in range(len(scores)) if j != exclude_idx
    ]
    if not candidates:
        return None

    # Score each candidate by (bucket_deficit, -reward) so we prefer
    # under-represented buckets and, within the same bucket, worse outputs.
    min_count = min(bucket_counts[b] for b in BUCKETS)

    def sort_key(j):
        mode = classify_rejected_mode(scores[j])
        deficit = bucket_counts[mode] - min_count  # lower = more under-represented
        return (deficit, rewards[j])               # ascending: prefer low deficit, low reward

    return min(candidates, key=sort_key)


pairs = []
skipped_low_gap = 0
skipped_no_valid = 0
bucket_counts = {b: 0 for b in BUCKETS}

for i, row in enumerate(rows):
    template   = row['template']
    label_key  = REGISTRY[template]['label_key']
    gold       = row[label_key]
    transcript = row['transcript']

    samples = sample_n(template, transcript)
    scores  = [score_prediction(template, s, gold, transcript) for s in samples]
    rewards = [reward(sc) for sc in scores]

    valid_idxs = [j for j in range(len(scores)) if scores[j]['schema_valid']]

    if not valid_idxs:
        skipped_no_valid += 1
        continue

    best_idx = max(range(len(rewards)), key=lambda j: rewards[j])

    # Compute gold reward honestly (do not hardcode 1.0).
    gold_raw     = json.dumps(gold, ensure_ascii=False)
    gold_score   = score_prediction(template, gold_raw, gold, transcript)
    reward_gold  = reward(gold_score)

    # Hybrid chosen: best model sample if it clears threshold, else gold.
    if rewards[best_idx] >= QUALITY_THRESHOLD and scores[best_idx]['schema_valid']:
        chosen_text   = samples[best_idx]
        reward_chosen = rewards[best_idx]
        chosen_source = 'model'
    else:
        chosen_text   = gold_raw
        reward_chosen = reward_gold
        chosen_source = 'gold'

    # Build prompt string once (shared across both pairs for this transcript).
    messages   = build_inference_messages(template, transcript)
    prompt_str = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    chosen_idx = best_idx if chosen_source == 'model' else None

    # --- Primary rejected pair ---
    rej_idx = pick_rejected_idx(scores, rewards, chosen_idx, bucket_counts)
    if rej_idx is None:
        skipped_no_valid += 1
        continue

    gap = reward_chosen - rewards[rej_idx]
    if gap < MIN_GAP:
        skipped_low_gap += 1
        continue

    mode = classify_rejected_mode(scores[rej_idx])
    bucket_counts[mode] += 1

    pairs.append({
        'prompt':          prompt_str,
        'chosen':          chosen_text,
        'rejected':        samples[rej_idx],
        'template':        template,
        'reward_chosen':   round(reward_chosen, 4),
        'reward_rejected': round(rewards[rej_idx], 4),
        'reward_gold':     round(reward_gold, 4),
        'rejected_mode':   mode,
        'chosen_source':   chosen_source,
    })

    # --- Optional second rejected pair when gap is large ---
    # Picks a different sample to double training signal cheaply.
    if gap > LARGE_GAP:
        rej2_idx = pick_rejected_idx(scores, rewards, rej_idx if chosen_idx is None else chosen_idx, bucket_counts)
        if rej2_idx is not None and rej2_idx != rej_idx:
            gap2 = reward_chosen - rewards[rej2_idx]
            if gap2 >= MIN_GAP:
                mode2 = classify_rejected_mode(scores[rej2_idx])
                bucket_counts[mode2] += 1
                pairs.append({
                    'prompt':          prompt_str,
                    'chosen':          chosen_text,
                    'rejected':        samples[rej2_idx],
                    'template':        template,
                    'reward_chosen':   round(reward_chosen, 4),
                    'reward_rejected': round(rewards[rej2_idx], 4),
                    'reward_gold':     round(reward_gold, 4),
                    'rejected_mode':   mode2,
                    'chosen_source':   chosen_source,
                })

    if (i + 1) % 25 == 0:
        print(f'  [{i+1}/{len(rows)}] pairs: {len(pairs)}, '
              f'skipped low-gap: {skipped_low_gap}, no-valid: {skipped_no_valid}')

print(f'\nDone. pairs={len(pairs)}, skipped_low_gap={skipped_low_gap}, skipped_no_valid={skipped_no_valid}')
print('bucket_counts:', bucket_counts)

## Cell 8: balance check and save

In [ ]:
from collections import Counter

print(f'Total pairs: {len(pairs)}')
print()

# Per template
tpl_counts = Counter(p['template'] for p in pairs)
print('| template | count |')
print('|---|---|')
for t, c in sorted(tpl_counts.items()):
    print(f'| {t} | {c} |')
print()

# Per rejected mode — target ~25% each across four buckets
mode_counts = Counter(p['rejected_mode'] for p in pairs)
total = len(pairs)
print('| rejected_mode | count | % |')
print('|---|---|---|')
for m in ('miss', 'over_populated', 'hallucination', 'schema_invalid'):
    c   = mode_counts.get(m, 0)
    pct = 100 * c / total if total else 0
    warn = '  <-- WARNING: under-represented (<15%)' if pct < 15 else ''
    print(f'| {m} | {c} | {pct:.1f}%{warn} |')
print()

# Chosen source
src_counts = Counter(p['chosen_source'] for p in pairs)
print('chosen source:', dict(src_counts))
print()

# Reward gap stats
gaps = [p['reward_chosen'] - p['reward_rejected'] for p in pairs]
print(f'reward gap  mean={sum(gaps)/len(gaps):.3f}  min={min(gaps):.3f}  max={max(gaps):.3f}')

# Gold reward distribution (sanity: should be mostly high but not always 1.0)
gold_rewards = [p['reward_gold'] for p in pairs]
print(f'reward_gold mean={sum(gold_rewards)/len(gold_rewards):.3f}  '
      f'min={min(gold_rewards):.3f}  max={max(gold_rewards):.3f}')
print()

# Save
with open(OUT_PATH, 'w') as f:
    for p in pairs:
        f.write(json.dumps(p, ensure_ascii=False) + '\n')
print(f'saved -> {OUT_PATH}  ({len(pairs)} lines)')